# Lab 03 — Tensors: The Core of PyTorch

**Topic:** Creating, reshaping, indexing, and operating on tensors  
**Course:** PyTorch for Deep Learning Professional Certificate — DeepLearning.AI

> This is my personal study notebook based on the tensor concepts covered in the course lab.
> I rewrote the explanations in my own words and added a few small experiments so I can use this notebook later as a practical PyTorch reference.

## What I wanted to understand

For me, the main goal of this lab was not just learning individual tensor functions. I wanted to get comfortable with three things that seem to cause a lot of PyTorch bugs:

1. **shape** — what dimensions my data has,
2. **dtype** — what type the values are stored as,
3. **device/layout compatibility** — whether tensors can actually participate in the same operation.

In this notebook I focus mostly on shape and dtype, because they show up everywhere once model building starts.

## Setup

In [1]:
import torch
import numpy as np
import pandas as pd

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cpu


## 1. Creating tensors

A tensor is the main data structure PyTorch uses for numerical computation.  
My simplest mental model is: **a tensor is like a NumPy array that is designed to work naturally with deep learning operations and accelerators.**

I can create tensors from existing Python/NumPy data or generate them directly.

In [2]:
# From a Python list
from_list = torch.tensor([1, 2, 3])

# From a NumPy array
np_array = np.array([[1, 2, 3],
                     [4, 5, 6]])
from_numpy = torch.from_numpy(np_array)

print("From list:", from_list)
print("dtype:", from_list.dtype)

print("\nFrom NumPy:")
print(from_numpy)
print("shape:", from_numpy.shape, "| dtype:", from_numpy.dtype)

From list: tensor([1, 2, 3])
dtype: torch.int64

From NumPy:
tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3]) | dtype: torch.int64


### DataFrame → tensor

There is no special `DataFrame -> Tensor` function that I need to memorize.  
The practical path is simply:

`DataFrame -> NumPy values -> torch.tensor(...)`

I use an inline DataFrame here so this notebook stays self-contained when I push it to GitHub.

In [3]:
df = pd.DataFrame({
    "feature_1": [1.2, 2.4, 3.1],
    "feature_2": [10.0, 12.5, 9.8]
})

tensor_from_df = torch.tensor(df.values, dtype=torch.float32)

print(df)
print("\nTensor:")
print(tensor_from_df)
print("shape:", tensor_from_df.shape, "| dtype:", tensor_from_df.dtype)

   feature_1  feature_2
0        1.2       10.0
1        2.4       12.5
2        3.1        9.8

Tensor:
tensor([[ 1.2000, 10.0000],
        [ 2.4000, 12.5000],
        [ 3.1000,  9.8000]])
shape: torch.Size([3, 2]) | dtype: torch.float32


### Creating tensors with known values

These functions are useful for initialization, testing shapes, or quickly generating sample data.

In [4]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
random_values = torch.rand(2, 3)
sequence = torch.arange(0, 10, step=2)

print("zeros:\n", zeros)
print("\nones:\n", ones)
print("\nrandom:\n", random_values)
print("\nsequence:", sequence)

zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])

ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

random:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])

sequence: tensor([0, 2, 4, 6, 8])


## 2. Shapes and reshaping

This section is probably the most important part of the lab for me.

A tensor can contain the correct numbers and still fail in a model because its **shape is wrong**.  
So before debugging anything more complicated, I should check `.shape`.

In [5]:
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])

print(x)
print("shape:", x.shape)
print("number of dimensions:", x.ndim)
print("number of elements:", x.numel())

tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3])
number of dimensions: 2
number of elements: 6


### `unsqueeze()` and `squeeze()`

`unsqueeze()` adds a dimension of size 1.  
This is especially useful when I have one sample but a model expects a batch.

`squeeze()` removes dimensions whose size is 1.

In [6]:
expanded = x.unsqueeze(0)
restored = expanded.squeeze(0)

print("Original shape:", x.shape)
print("After unsqueeze(0):", expanded.shape)
print("After squeeze(0):", restored.shape)

Original shape: torch.Size([2, 3])
After unsqueeze(0): torch.Size([1, 2, 3])
After squeeze(0): torch.Size([2, 3])


### `reshape()`

`reshape()` changes how the same values are organized without changing the total number of elements.

A useful check is:

**old number of elements == new number of elements**

In [7]:
flat = x.reshape(-1)
reshaped = flat.reshape(3, 2)

print("Original:\n", x)
print("\nFlattened:", flat, "| shape:", flat.shape)
print("\nReshaped to 3 x 2:\n", reshaped)

Original:
 tensor([[1, 2, 3],
        [4, 5, 6]])

Flattened: tensor([1, 2, 3, 4, 5, 6]) | shape: torch.Size([6])

Reshaped to 3 x 2:
 tensor([[1, 2],
        [3, 4],
        [5, 6]])


### Transpose

Transposing swaps dimensions. For a 2D tensor, it is similar to transposing a matrix.

In [8]:
transposed = x.transpose(0, 1)

print("Original shape:", x.shape)
print(x)

print("\nTransposed shape:", transposed.shape)
print(transposed)

Original shape: torch.Size([2, 3])
tensor([[1, 2, 3],
        [4, 5, 6]])

Transposed shape: torch.Size([3, 2])
tensor([[1, 4],
        [2, 5],
        [3, 6]])


### Combining tensors

`torch.cat()` joins tensors along an existing dimension.

The key thing I need to check first is that the **other dimensions are compatible**.

In [9]:
a = torch.tensor([[1, 2],
                  [3, 4]])

b = torch.tensor([[5, 6]])

combined_rows = torch.cat((a, b), dim=0)

print("A:\n", a)
print("\nB:\n", b)
print("\nConcatenated along rows:\n", combined_rows)
print("shape:", combined_rows.shape)

A:
 tensor([[1, 2],
        [3, 4]])

B:
 tensor([[5, 6]])

Concatenated along rows:
 tensor([[1, 2],
        [3, 4],
        [5, 6]])
shape: torch.Size([3, 2])


## 3. Indexing and slicing

Tensor indexing feels very similar to NumPy, which is useful because I already use NumPy/pandas frequently.

The pattern I want to remember is:

`tensor[rows, columns]`

In [10]:
x = torch.tensor([
    [10, 20, 30, 40],
    [50, 60, 70, 80],
    [90, 100, 110, 120]
])

second_row = x[1]
third_column = x[:, 2]
middle_block = x[0:2, 1:3]
single_value = x[2, 3]

print("Tensor:\n", x)
print("\nSecond row:", second_row)
print("Third column:", third_column)
print("Middle block:\n", middle_block)
print("Single value:", single_value.item())

Tensor:
 tensor([[ 10,  20,  30,  40],
        [ 50,  60,  70,  80],
        [ 90, 100, 110, 120]])

Second row: tensor([50, 60, 70, 80])
Third column: tensor([ 30,  70, 110])
Middle block:
 tensor([[20, 30],
        [60, 70]])
Single value: 120


### Boolean masking

This is one of the tensor operations I expect to use a lot in data preparation.  
A comparison creates a boolean tensor, and that boolean tensor can be used as a mask.

In [11]:
values = torch.tensor([12, 35, 18, 42, 27, 50])

mask = values > 30
selected = values[mask]

print("Values:", values)
print("Mask:  ", mask)
print("Values > 30:", selected)

Values: tensor([12, 35, 18, 42, 27, 50])
Mask:   tensor([False,  True, False,  True, False,  True])
Values > 30: tensor([35, 42, 50])


## 4. Mathematical operations

There is an important distinction between:

- **element-wise multiplication:** `a * b`
- **matrix/vector multiplication:** `torch.matmul(a, b)` or `a @ b`

They may look similar in code, but mathematically they are different operations.

In [12]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])

print("a + b =", a + b)
print("a * b =", a * b)
print("dot product =", torch.matmul(a, b))

a + b = tensor([5., 7., 9.])
a * b = tensor([ 4., 10., 18.])
dot product = tensor(32.)


### Broadcasting

Broadcasting lets PyTorch operate on tensors with different but compatible shapes.

Instead of manually copying a smaller tensor, PyTorch behaves *as if* it were expanded to match the larger shape.

In [13]:
row = torch.tensor([1, 2, 3])          # shape [3]
column = torch.tensor([[10],
                       [20],
                       [30]])            # shape [3, 1]

result = row + column

print("row shape:", row.shape)
print("column shape:", column.shape)
print("result shape:", result.shape)
print("\nresult:\n", result)

row shape: torch.Size([3])
column shape: torch.Size([3, 1])
result shape: torch.Size([3, 3])

result:
 tensor([[11, 12, 13],
        [21, 22, 23],
        [31, 32, 33]])


### My broadcasting check

When broadcasting confuses me, I can compare tensor shapes from the **right-most dimension backward**.  
Dimensions are compatible when they are equal or when one of them is `1`.

In [14]:
features = torch.tensor([
    [10., 20., 30.],
    [40., 50., 60.]
])

feature_scale = torch.tensor([0.1, 1.0, 10.0])

scaled_features = features * feature_scale

print("features shape:", features.shape)
print("scale shape:   ", feature_scale.shape)
print("\nScaled features:\n", scaled_features)

features shape: torch.Size([2, 3])
scale shape:    torch.Size([3])

Scaled features:
 tensor([[  1.,  20., 300.],
        [  4.,  50., 600.]])


## 5. Comparisons, statistics, and dtypes

Boolean logic is useful for feature engineering and filtering.  
Statistics are useful for inspecting data and later for normalization.

In [15]:
temperatures = torch.tensor([20., 35., 19., 35., 42.])

is_hot = temperatures > 30
is_35 = temperatures == 35
comfortable = (temperatures >= 20) & (temperatures <= 35)

print("temperatures:", temperatures)
print("hot:", is_hot)
print("exactly 35:", is_35)
print("20 to 35:", comfortable)

print("\nMean:", temperatures.mean().item())
print("Std:", temperatures.std().item())

temperatures: tensor([20., 35., 19., 35., 42.])
hot: tensor([False,  True, False,  True,  True])
exactly 35: tensor([False,  True, False,  True, False])
20 to 35: tensor([ True,  True, False,  True, False])

Mean: 30.200000762939453
Std: 10.183320045471191


### Dtypes matter

Most neural-network computations use floating-point tensors.  
If a model expects `float32` and I pass integer data, I can run into errors.

So I should get used to checking both:

```python
tensor.shape
tensor.dtype
```

In [16]:
integer_tensor = torch.tensor([10, 20, 30])
float_tensor = integer_tensor.float()

print("Before:", integer_tensor, integer_tensor.dtype)
print("After: ", float_tensor, float_tensor.dtype)

Before: tensor([10, 20, 30]) torch.int64
After:  tensor([10., 20., 30.]) torch.float32


## 6. Small practical exercise — feature engineering with tensors

I wanted one example that feels closer to a normal data-science workflow.

Suppose each row represents a taxi trip:

`[distance_km, hour_of_day]`

I create a new feature that marks a trip as `1` only when it is:

- longer than 10 km, **and**
- during morning rush hour `[8, 10)` or evening rush hour `[17, 19)`.

In [17]:
trip_data = torch.tensor([
    [5.3,  7],
    [12.1, 9],
    [15.5, 13],
    [6.7, 18],
    [2.4, 20],
    [11.8, 17],
    [9.0,  9],
    [14.2, 8]
], dtype=torch.float32)

distance = trip_data[:, 0]
hour = trip_data[:, 1]

long_trip = distance > 10
morning_rush = (hour >= 8) & (hour < 10)
evening_rush = (hour >= 17) & (hour < 19)

rush_hour_long_trip = long_trip & (morning_rush | evening_rush)

# Convert the boolean feature to float and reshape it into a column.
new_feature = rush_hour_long_trip.float().unsqueeze(1)

enhanced_trip_data = torch.cat((trip_data, new_feature), dim=1)

print("New feature:", rush_hour_long_trip)
print("\nEnhanced data [distance, hour, rush_hour_long_trip]:\n")
print(enhanced_trip_data)

New feature: tensor([False,  True, False, False, False,  True, False,  True])

Enhanced data [distance, hour, rush_hour_long_trip]:

tensor([[ 5.3000,  7.0000,  0.0000],
        [12.1000,  9.0000,  1.0000],
        [15.5000, 13.0000,  0.0000],
        [ 6.7000, 18.0000,  0.0000],
        [ 2.4000, 20.0000,  0.0000],
        [11.8000, 17.0000,  1.0000],
        [ 9.0000,  9.0000,  0.0000],
        [14.2000,  8.0000,  1.0000]])


## 7. One more shape experiment: image batches

For image data, dimensions have specific meanings.

A grayscale image batch may start as:

`[batch, height, width]`

If a model expects an explicit channel dimension, I can add one:

`[batch, channel, height, width]`

In [18]:
images = torch.rand(4, 3, 2)

images_nchw = images.unsqueeze(1)
images_nhwc = images_nchw.permute(0, 2, 3, 1)

print("Original [N, H, W]:", images.shape)
print("With channel [N, C, H, W]:", images_nchw.shape)
print("Channel last [N, H, W, C]:", images_nhwc.shape)

Original [N, H, W]: torch.Size([4, 3, 2])
With channel [N, C, H, W]: torch.Size([4, 1, 3, 2])
Channel last [N, H, W, C]: torch.Size([4, 3, 2, 1])


## What I am taking away from this lab

The biggest takeaway for me is that tensors are not difficult because of the arithmetic itself.  
The difficult part is keeping track of **what each dimension represents**.

My practical checklist before sending data into a model is now:

- Check the tensor's **shape**.
- Check the **dtype**.
- Know what each axis means.
- Use `unsqueeze`, `reshape`, `transpose`/`permute`, or `cat` intentionally.
- Remember that broadcasting can be convenient, but I still need to understand why the shapes are compatible.
- For debugging, print shapes early instead of waiting for a later model error.

This lab also made the connection to neural networks clearer: batches, features, weights, matrix multiplication, and broadcasting are all tensor operations. Once these operations feel natural, the model code becomes much easier to reason about.

---

### Course reference

Concepts practiced here were based on **Course 1, Module 1 — Lab 3: Tensors** from the  
**PyTorch for Deep Learning Professional Certificate (DeepLearning.AI)**.

This notebook is my rewritten study version for review and portfolio documentation.